In [ ]:
# ==============================================================================
# Block 1: CNN Encoder + Autoregressive Transformer Decoder
# ==============================================================================

import torch
import torch.nn as nn


class AutoregressiveTokenDecoder(nn.Module):
    """
    用於 2~10 層薄膜反向設計的 Autoregressive Transformer。

    Input
    -----
    spectrum:
        Transmittance Spectrum
        shape = (batch_size, 601)

    tgt_tokens:
        已知的結構 Token 序列
        shape = (batch_size, sequence_length)

    Output
    ------
    logits:
        每個位置對 294 個 Token 的預測結果
        shape = (batch_size, sequence_length, 294)

    Token 定義
    ----------
    Token 0:
        <BOS>

    Token 1:
        <PAD>，代表不存在的薄膜層

    Token 2~147:
        eps = 11.7，厚度 10~300 nm，間隔 2 nm

    Token 148~293:
        eps = 3.9，厚度 10~300 nm，間隔 2 nm
    """

    def __init__(
        self,
        input_dim=601,
        num_layers=10,
        vocab_size=294,
        d_model=256,
        nhead=8,
        num_layers_dec=6,
        dim_feedforward=1024,
        dropout=0.05,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.num_layers = num_layers
        self.d_model = d_model

        # 1D-CNN Transmittance Spectrum 特徵抽取器
        # (B, 1, 601) -> (B, 256, 601)
        self.spec_conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Conv1d(128, d_model, kernel_size=3, padding=1),
        )

        # 601 個波長位置的 Positional Embedding
        self.spec_pos_emb = nn.Parameter(torch.randn(1, input_dim, d_model) * 0.02)

        # Structure Token Embedding 與位置資訊
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, num_layers + 1, d_model) * 0.02)

        # 6 層 Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )

        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers_dec)

        # (B, seq_len, 256) -> (B, seq_len, 294)
        self.head = nn.Linear(d_model, vocab_size)


    def forward(self, spectrum, tgt_tokens):

        # Transmittance Spectrum -> CNN Features
        # (B, 601) -> (B, 1, 601) -> (B, 256, 601) -> (B, 601, 256)
        spec_feats = self.spec_conv(spectrum.unsqueeze(1)).transpose(1, 2)
        spec_feats = spec_feats + self.spec_pos_emb

        # Structure Token -> Embedding
        # (B, seq_len) -> (B, seq_len, 256)
        _, seq_len = tgt_tokens.size()
        tgt_emb = self.token_emb(tgt_tokens) + self.pos_emb[:, :seq_len, :]

        # 防止目前位置看到後面的正確 Token
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(spectrum.device)

        # Decoder 使用 CNN Spectrum Features 作為 memory
        out = self.transformer_decoder(tgt=tgt_emb, memory=spec_feats, tgt_mask=causal_mask)

        # 每個位置輸出 294 個 Token logits
        logits = self.head(out)

        return logits

In [ ]:
# ==============================================================================
# Block 2: Dataset Loading + Training
# ==============================================================================

import os
import glob
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


"""
訓練參數設定
"""

DATA_DIR = "IntegerData_VarLayers"

BATCH_SIZE = 128
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
TOPOLOGY_WEIGHT = 1.5

TRAIN_RATIO = 0.9
SPLIT_SEED = 42

MODEL_SAVE_PATH = "best_ar_token_decoder_variable.pth"
PLOT_SAVE_PATH = "training_curve_variable.png"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if os.name == "nt" else 4

print(f"Device: {DEVICE}")


class ThinFilmTokenizer:
    """
    將材料與厚度轉換成 Token。

    Token 0:
        <BOS>

    Token 1:
        <PAD>，代表不存在的薄膜層

    Token 2~147:
        eps = 11.7，厚度 10~300 nm，間隔 2 nm

    Token 148~293:
        eps = 3.9，厚度 10~300 nm，間隔 2 nm
    """

    def __init__(self):
        self.vocab_size = 294
        self.bos_token = 0
        self.pad_token = 1

    def encode_layer(self, material, thickness):

        # 厚度接近 0 nm 表示此層不存在
        if thickness < 5.0:
            return self.pad_token

        thickness = np.clip(thickness, 10.0, 300.0)
        thickness_idx = int(np.round((thickness - 10.0) / 2.0))
        thickness_idx = np.clip(thickness_idx, 0, 145)

        if abs(material - 11.7) < abs(material - 3.9):
            return 2 + thickness_idx

        else:
            return 148 + thickness_idx

    def decode_token(self, token):

        if token in (0, 1):
            return 0.0, 0.0

        elif 2 <= token <= 147:
            thickness = 10.0 + (token - 2) * 2.0
            return 11.7, thickness

        elif 148 <= token <= 293:
            thickness = 10.0 + (token - 148) * 2.0
            return 3.9, thickness

        return 0.0, 0.0


tokenizer = ThinFilmTokenizer()


class MultiMaterialFilmDataset(Dataset):
    """
    讀取 2~10 層薄膜 Dataset。

    Dataset 中：
        params : shape = (N, 20)
        T      : shape = (N, 601)

    params 格式：
        [eps1, d1, eps2, d2, ..., eps10, d10]

    每筆資料輸出：
        spectrum      : shape = (601,)
        decoder_input : shape = (10,)
        target_tokens : shape = (10,)
    """

    def __init__(self, file_paths):

        if isinstance(file_paths, str):
            file_paths = [file_paths]

        transmittance_list = []
        params_list = []

        print(f"Loading {len(file_paths)} Dataset shards...")

        for file_path in file_paths:
            data = np.load(file_path)

            transmittance_list.append(data["T"])
            params_list.append(data["params"])

        self.transmittance = np.concatenate(transmittance_list, axis=0).astype(np.float32)
        params = np.concatenate(params_list, axis=0).astype(np.float32)

        self.materials = params[:, 0::2]
        self.thicknesses = params[:, 1::2]

        print(f"Dataset samples: {len(self.transmittance):,}")
        print(f"Spectrum dimension: {self.transmittance.shape[1]}")

    def __len__(self):
        return len(self.transmittance)

    def __getitem__(self, idx):

        spectrum = self.transmittance[idx]
        materials = self.materials[idx]
        thicknesses = self.thicknesses[idx]

        target_tokens = [
            tokenizer.encode_layer(material, thickness)
            for material, thickness in zip(materials, thicknesses)
        ]

        decoder_input = [tokenizer.bos_token] + target_tokens[:-1]

        return (
            torch.tensor(spectrum, dtype=torch.float32),
            torch.tensor(decoder_input, dtype=torch.long),
            torch.tensor(target_tokens, dtype=torch.long),
        )


class StructuralAwareLoss(nn.Module):
    """
    Loss = Cross Entropy + topology_weight × Topology Loss

    Cross Entropy：
        判斷完整的材料與厚度 Token 是否正確。

    Topology Loss：
        額外強化模型判斷該位置是有效薄膜層或 <PAD>，
        用來提升 2~10 層結構的層數判斷能力。
    """

    def __init__(
        self,
        vocab_size=294,
        topology_weight=1.5,
        label_smoothing=0.05,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.topology_weight = topology_weight

        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.bce = nn.BCEWithLogitsLoss()

        self.non_pad_indices = list(range(2, vocab_size))

    def forward(self, logits, targets):

        flat_logits = logits.reshape(-1, self.vocab_size)
        flat_targets = targets.reshape(-1)

        # 一般 Token Cross Entropy
        loss_ce = self.ce(flat_logits, flat_targets)

        # PAD = 0，有效薄膜層 = 1
        target_is_layer = (flat_targets != tokenizer.pad_token).float()

        pad_logits = flat_logits[:, tokenizer.pad_token]
        layer_logits = torch.logsumexp(flat_logits[:, self.non_pad_indices], dim=-1)

        # 判斷該位置是 PAD 還是真正的薄膜層
        loss_topology = self.bce(layer_logits - pad_logits, target_is_layer)

        loss = loss_ce + self.topology_weight * loss_topology

        return loss, loss_ce, loss_topology


"""
尋找 Dataset 並建立 DataLoader
"""

data_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "Data_*.npz"), recursive=True))

if len(data_files) == 0:
    raise FileNotFoundError(f"在 {DATA_DIR} 找不到 Data_*.npz")

print(f"Found {len(data_files)} Dataset shards")

full_dataset = MultiMaterialFilmDataset(data_files)

train_size = int(TRAIN_RATIO * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SPLIT_SEED),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)


"""
建立模型、Loss、Optimizer 與 Learning Rate Scheduler
"""

input_dim = full_dataset.transmittance.shape[1]

model = AutoregressiveTokenDecoder(
    input_dim=input_dim,
    vocab_size=tokenizer.vocab_size,
).to(DEVICE)

criterion = StructuralAwareLoss(
    vocab_size=tokenizer.vocab_size,
    topology_weight=TOPOLOGY_WEIGHT,
    label_smoothing=LABEL_SMOOTHING,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6,
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


"""
Training
"""

history = {
    "train_loss": [],
    "train_ce": [],
    "train_topology": [],
    "val_loss": [],
    "val_accuracy": [],
}

best_val_loss = float("inf")


for epoch in range(1, EPOCHS + 1):

    model.train()

    total_loss = 0.0
    total_ce = 0.0
    total_topology = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{EPOCHS:02d}]")

    for spectrum, decoder_input, targets in pbar:

        spectrum = spectrum.to(DEVICE, non_blocking=True)
        decoder_input = decoder_input.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(spectrum, decoder_input)
            loss, loss_ce, loss_topology = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_ce += loss_ce.item()
        total_topology += loss_topology.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}")


    scheduler.step()


    """
    Validation
    """

    model.eval()

    val_loss = 0.0
    correct_tokens = 0
    total_tokens = 0

    with torch.no_grad():

        for spectrum, decoder_input, targets in val_loader:

            spectrum = spectrum.to(DEVICE, non_blocking=True)
            decoder_input = decoder_input.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits = model(spectrum, decoder_input)
                loss, _, _ = criterion(logits, targets)

            val_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)

            correct_tokens += (predictions == targets).sum().item()
            total_tokens += targets.numel()


    avg_train_loss = total_loss / len(train_loader)
    avg_train_ce = total_ce / len(train_loader)
    avg_train_topology = total_topology / len(train_loader)

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct_tokens / total_tokens


    history["train_loss"].append(avg_train_loss)
    history["train_ce"].append(avg_train_ce)
    history["train_topology"].append(avg_train_topology)
    history["val_loss"].append(avg_val_loss)
    history["val_accuracy"].append(val_accuracy)


    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"CE: {avg_train_ce:.4f} | "
        f"Topology: {avg_train_topology:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy * 100:.2f}%"
    )


    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        torch.save(
            model.state_dict(),
            MODEL_SAVE_PATH,
        )

        print(f"Saved best model | Val Loss: {best_val_loss:.4f}")


print("\nTraining Finished")


"""
Training Curve
"""

epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)


plt.subplot(1, 2, 2)
plt.plot(epochs, history["val_accuracy"], label="Validation Token Accuracy")

plt.title("Validation Token Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.savefig(PLOT_SAVE_PATH, dpi=300)
plt.show()

In [ ]:
# ==============================================================================
# Block 3: Export Trained Model
# ==============================================================================

import torch


"""
將 Validation Loss 最低的模型權重，
連同模型架構與 Tokenizer 設定一起打包。

輸出的 .pth 可以在其他電腦重新建立模型後，
直接載入權重進行 inference，不需要重新訓練。
"""

EXPORT_PATH = "thin_film_transformer_variable_final.pth"

# 載入 Block 2 儲存的最佳模型權重
best_state_dict = torch.load(MODEL_SAVE_PATH, map_location="cpu")

export_payload = {
    "model_config": {
        "input_dim": full_dataset.transmittance.shape[1],
        "num_layers": 10,
        "vocab_size": 294,
        "d_model": 256,
        "nhead": 8,
        "num_layers_dec": 6,
        "dim_feedforward": 1024,
        "dropout": 0.05,
    },

    "tokenizer_config": {
        "bos_token": 0,
        "pad_token": 1,
        "eps_values": [11.7, 3.9],
        "thickness_min_nm": 10.0,
        "thickness_max_nm": 300.0,
        "thickness_step_nm": 2.0,
    },

    "state_dict": best_state_dict,
}

torch.save(export_payload, EXPORT_PATH)

print(f"Model exported to: {EXPORT_PATH}")